In [2]:
ls

 Volume in drive C has no label.
 Volume Serial Number is C616-29CA

 Directory of C:\Users\Lenovo\MSc thesis

01-12-2025  11:07    <DIR>          .
01-12-2025  11:06    <DIR>          ..
01-12-2025  11:07    <DIR>          .ipynb_checkpoints
01-12-2025  11:06               337 make_library.ipynb
               1 File(s)            337 bytes
               3 Dir(s)  34,354,229,248 bytes free


In [5]:
# =======================
# SIMPLE DNA LIBRARY MAKER
# =======================

# >>>>>> EDIT THESE TWO ONLY <<<<<<
PARENT_SEQ = "GCGTGAAGAGGGTAGTCCGACATCGGGAGG"   # put YOUR parent sequence here
TARGET_LEN = 30                 # put NUMBER OF BASES you want (e.g. 14, 16, etc.)
TOTAL_COUNT = 1000              # number of sequences to generate
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

import random
import csv

BASES = ['A','T','G','C']
MAX_HOMOPOLYMER = 6

def normalize(seq, L):
    seq = seq.upper().strip()
    if len(seq) >= L:
        return seq[:L]
    else:
        pad = ''.join(random.choice(BASES) for _ in range(L - len(seq)))
        return seq + pad

def has_long_run(seq, max_run=MAX_HOMOPOLYMER):
    run = 1
    last = seq[0]
    for c in seq[1:]:
        if c == last:
            run += 1
            if run > max_run:
                return True
        else:
            run = 1
            last = c
    return False

def scramble(seq):
    s = list(seq)
    random.shuffle(s)
    return ''.join(s)

def point_mut(seq):
    s = list(seq)
    pos = random.randrange(len(s))
    choices = [b for b in BASES if b != s[pos]]
    s[pos] = random.choice(choices)
    return ''.join(s)

def random_seq(n):
    return ''.join(random.choice(BASES) for _ in range(n))

# --------------------
# BUILD LIBRARY
# --------------------
parent = normalize(PARENT_SEQ, TARGET_LEN)

library = [parent]  # include parent first

while len(library) < TOTAL_COUNT:
    r = random.random()

    if r < 0.3:
        seq = scramble(parent)
    elif r < 0.6:
        seq = point_mut(parent)
    elif r < 0.8:
        seq = random_seq(TARGET_LEN)
    elif r < 0.9:
        seq = ''.join(random.choices(['G','C','A','T'], weights=[0.35,0.35,0.15,0.15], k=TARGET_LEN))
    else:
        seq = ''.join(random.choices(['A','T','G','C'], weights=[0.4,0.4,0.1,0.1], k=TARGET_LEN))

    if seq not in library and not has_long_run(seq):
        library.append(seq)

# --------------------
# SAVE TO FASTA
# --------------------
with open("library.fasta", "w") as f:
    for i, seq in enumerate(library, start=1):
        f.write(f">seq{i:05d}\n{seq}\n")

print("Library created: library.fasta")
print(f"Total sequences: {len(library)}")

Library created: library.fasta
Total sequences: 1000


In [6]:
# ===============================
# SIMPLE RNA LIBRARY MAKER (1000)
# ===============================
# Edit only these three lines:
PARENT_SEQ = "AGGUACGACGGUUGGAGGAGUUCGAGUCCG"   # <- PUT your parent RNA sequence here (A U G C only)
TARGET_LEN = 30                # <- number of bases to trim/pad to (e.g., 14, 16)
TOTAL_COUNT = 1000             # <- number of RNA sequences you want

# (You can change the three lines above and re-run the cell.)

import random
import csv

BASES = ['A','U','G','C']
MAX_HOMOPOLYMER = 6
AVOID_PALINDROME = True  # keeps self-complementary sequences out

# ---------- helper funcs ----------
def normalize(seq, L):
    s = seq.strip().upper().replace(" ", "")
    if len(s) >= L:
        return s[:L]
    else:
        pad = ''.join(random.choice(BASES) for _ in range(L - len(s)))
        return s + pad

def has_long_run(seq, max_run=MAX_HOMOPOLYMER):
    run = 1
    last = seq[0]
    for c in seq[1:]:
        if c == last:
            run += 1
            if run > max_run:
                return True
        else:
            run = 1
            last = c
    return False

def rc_rna(seq):
    # reverse complement for RNA: A<->U, G<->C
    comp = {'A':'U','U':'A','G':'C','C':'G'}
    return ''.join(comp.get(b,'N') for b in reversed(seq))

def is_palindrome(seq):
    return rc_rna(seq) == seq

def scramble(seq):
    s = list(seq)
    random.shuffle(s)
    return ''.join(s)

def point_mut(seq, nmut=1):
    s = list(seq)
    L = len(s)
    for _ in range(nmut):
        i = random.randrange(L)
        choices = [b for b in BASES if b != s[i]]
        s[i] = random.choice(choices)
    return ''.join(s)

def small_random_mutation(seq, prob=0.12):
    s = []
    for ch in seq:
        if random.random() < prob:
            s.append(random.choice([b for b in BASES if b != ch]))
        else:
            s.append(ch)
    return ''.join(s)

def make_gc_rich(n):
    return ''.join(random.choices(['G','C','A','U'], weights=[0.35,0.35,0.15,0.15], k=n))

def make_au_rich(n):
    return ''.join(random.choices(['A','U','G','C'], weights=[0.4,0.4,0.1,0.1], k=n))

def random_seq(n):
    return ''.join(random.choice(BASES) for _ in range(n))

def acceptable(seq):
    if has_long_run(seq):
        return False
    if AVOID_PALINDROME and is_palindrome(seq):
        return False
    return True

# ---------- build library ----------
parent = normalize(PARENT_SEQ, TARGET_LEN)
if any(ch not in "AUGC" for ch in parent):
    raise SystemExit("Parent sequence must contain only A U G C (uppercase or lowercase ok).")

library = [parent]         # include parent as seq00001
meta = [("parent", parent)]

attempts = 0
max_attempts = TOTAL_COUNT * 30

while len(library) < TOTAL_COUNT and attempts < max_attempts:
    attempts += 1
    r = random.random()
    if r < 0.25:
        seq = scramble(parent)
        typ = "scrambled"
    elif r < 0.50:
        seq = point_mut(parent, nmut=1)
        typ = "pointmut_1"
    elif r < 0.70:
        seq = small_random_mutation(parent, prob=0.12)
        typ = "small_rand_mut"
    elif r < 0.82:
        seq = make_gc_rich(TARGET_LEN)
        typ = "gc_rich"
    elif r < 0.92:
        seq = make_au_rich(TARGET_LEN)
        typ = "au_rich"
    else:
        seq = random_seq(TARGET_LEN)
        typ = "random"

    if seq in library:
        continue
    if not acceptable(seq):
        continue

    library.append(seq)
    meta.append((typ, seq))

if len(library) < TOTAL_COUNT:
    print(f"Warning: only generated {len(library)} sequences (requested {TOTAL_COUNT})")

# ---------- write outputs ----------
FASTA_OUT = "library_rna.fasta"
META_OUT = "library_rna_meta.csv"

with open(FASTA_OUT, "w") as fa:
    for i, seq in enumerate(library, start=1):
        fa.write(f">seq{i:05d}|{meta[i-1][0]}\n{seq}\n")

with open(META_OUT, "w", newline="") as csvf:
    w = csv.writer(csvf)
    w.writerow(["id","type","seq","len"])
    for i, (typ, seq) in enumerate(meta, start=1):
        w.writerow([f"seq{i:05d}", typ, seq, len(seq)])

print("Wrote:", FASTA_OUT, "| count:", len(library))
print("Metadata:", META_OUT)

Wrote: library_rna.fasta | count: 1000
Metadata: library_rna_meta.csv


In [7]:
# ============================
# MAKE CSV FROM DNA FASTA ONLY
# ============================

import csv

FASTA_FILE = "dna_library.fasta"        # your existing fasta
CSV_FILE = "dna_library_meta.csv"       # output csv

def gc_fraction(seq):
    seq = seq.upper()
    gc = seq.count("G") + seq.count("C")
    return gc / len(seq)

ids = []
types = []
seqs = []
lengths = []
gcs = []

with open(FASTA_FILE, "r") as f:
    current_id = None
    current_type = None

    for line in f:
        line = line.strip()
        
        if line.startswith(">"):
            # Example header: >seq00001|random
            header = line[1:]  # remove ">"
            parts = header.split("|")
            current_id = parts[0]
            current_type = parts[1] if len(parts) > 1 else "unknown"
        
        else:
            seq = line.strip()
            ids.append(current_id)
            types.append(current_type)
            seqs.append(seq)
            lengths.append(len(seq))
            gcs.append(round(gc_fraction(seq), 3))

# write CSV
with open(CSV_FILE, "w", newline="") as csvf:
    w = csv.writer(csvf)
    w.writerow(["id", "type", "seq", "length", "gc_fraction"])

    for i in range(len(seqs)):
        w.writerow([ids[i], types[i], seqs[i], lengths[i], gcs[i]])

print("CSV created:", CSV_FILE)

CSV created: dna_library_meta.csv
